# 13 — Rigorous decoder audit (DistilGPT-2)

Compare **decoder** toxic scorers on the same Jigsaw slices:

| key | role |
|---|---|
| `distilgpt2-d1` | zero-shot yes/no (D1) |
| `distilgpt2-d2` | zero-shot fill-in-blank (D2) |
| `distilgpt2-d1-ft` | D1 fine-tuned on binary toxic (` yes` / ` no`) |

`distilgpt2-d1-ft` is **skipped** until Colab notebook 10 weights exist under
`models/distilgpt2-d1-ft/`. Decoder scoring is slow on CPU — keep `QUICK_MODE=True`
while iterating; lower `EVAL_N` further if needed.


In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

THRESHOLD = 0.5

_src = None
for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "paths.py").is_file():
        _src = str(_p / "src")
        break
if _src is None:
    raise FileNotFoundError("Cannot find DeepProjectHate/src/paths.py")
if _src not in sys.path:
    sys.path.insert(0, _src)

for _m in (
    "paths", "model_registry", "roster", "splits", "fairness",
    "perturb", "intervention", "device", "distill",
):
    sys.modules.pop(_m, None)

from device import get_device
from fairness import CONTROL_TEXTS, build_identity_probe_df, control_fpr, per_group_fpr
from intervention import mask_terms
from model_registry import ALL_SPECS, load_scorer
from paths import PROCESSED_DIR, PROJECT_ROOT, SEED
from perturb import PERTURBATIONS
from roster import model_ready
from splits import get_split_frame, sample_split_frame

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
device = get_device()
print("ROOT", PROJECT_ROOT)
print("device", device)

QUICK_MODE = True
EVAL_N = 40 if QUICK_MODE else 200
ROBUST_N = 30 if QUICK_MODE else 100
KEYS = ["distilgpt2-d1", "distilgpt2-d2", "distilgpt2-d1-ft"]
print(f"QUICK_MODE={QUICK_MODE} EVAL_N={EVAL_N}")
print("Roster:", KEYS)
print("Note: d1-ft skipped until models/distilgpt2-d1-ft exists.")

def binary_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    out = {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }
    if len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    else:
        out["roc_auc"] = float("nan")
        out["pr_auc"] = float("nan")
    return out


def score_split(df, scorer):
    texts = df["comment_text"].astype(str).tolist()
    y = df["toxic"].astype(int).values
    t0 = time.perf_counter()
    p = np.asarray(scorer.predict_toxic_proba(texts), dtype=float)
    ms = 1000.0 * (time.perf_counter() - t0) / max(len(texts), 1)
    m = binary_metrics(y, p)
    m["ms_per_comment"] = round(ms, 3)
    return p, y, m


def load_ready(keys):
    scorers = {}
    for key in keys:
        ok, detail = model_ready(key) if key in (
            "toxic-bert", "tfidf-svm", "distilgpt2-d1", "distilgpt2-d1-ft", "distilbert-student"
        ) else (True, "registry")
        # Extra checks for keys not in FAIR_ROSTER helpers
        if key == "distilbert-tox":
            ok, detail = True, "HF hub martin-ha/toxic-comment-model"
        if key == "distilgpt2-d2":
            ok, detail = model_ready("distilgpt2-d1")[0], "same DistilGPT-2 weights as d1"
        if not ok:
            print(f"  [SKIP] {key}: {detail}")
            continue
        try:
            scorers[key] = load_scorer(ALL_SPECS[key], device=device)
            n = getattr(scorers[key], "n_params", None)
            print(f"  [OK] {key}: {detail} | params={n}")
        except Exception as exc:  # noqa: BLE001
            print(f"  [SKIP] {key}: load failed: {exc}")
    return scorers


## 1. Load models (skip FT if missing)


In [ ]:
scorers = load_ready(KEYS)
if not scorers:
    raise RuntimeError("No decoder models loaded — download DistilGPT-2 first")
if "distilgpt2-d1-ft" not in scorers:
    print("\n*** distilgpt2-d1-ft not ready yet — continuing with zero-shot only ***")
val = sample_split_frame("val", n=EVAL_N, seed=SEED)
test = sample_split_frame("test", n=EVAL_N, seed=SEED)
print(f"val_n={len(val)} test_n={len(test)}")


## 2. Held-out metrics (val + test)


In [ ]:
metric_rows = []
probs = {}
for key, scorer in scorers.items():
    for split_name, df in (("val", val), ("test", test)):
        print(f"Scoring {key} / {split_name} ...")
        p, y, m = score_split(df, scorer)
        probs[(key, split_name)] = (p, y)
        metric_rows.append({"model": key, "split": split_name, **m,
                            "n_params": getattr(scorer, "n_params", None)})
        print(f"  F1={m['f1']:.3f} ROC-AUC={m['roc_auc']:.3f} ms={m['ms_per_comment']:.1f}")
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

test_m = metrics_df[metrics_df["split"] == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=test_m, x="model", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title("Test F1"); axes[0].tick_params(axis="x", rotation=20); axes[0].set_ylim(0, 1)
sns.barplot(data=test_m, x="model", y="roc_auc", ax=axes[1], color="seagreen")
axes[1].set_title("Test ROC-AUC"); axes[1].tick_params(axis="x", rotation=20); axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()


## 3. Score histograms (test) — degeneracy check


In [ ]:
n = len(scorers)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.2), sharey=True)
if n == 1:
    axes = [axes]
for ax, key in zip(axes, scorers):
    p, y = probs[(key, "test")]
    ax.hist(p[y == 0], bins=20, alpha=0.6, label="non-toxic", color="steelblue")
    ax.hist(p[y == 1], bins=20, alpha=0.6, label="toxic", color="crimson")
    ax.axvline(THRESHOLD, color="gray", ls="--")
    ax.set_title(key); ax.set_xlabel("P(toxic)"); ax.legend()
plt.tight_layout(); plt.show()


## 4. Error analysis (FP / FN samples)


In [ ]:
for key in scorers:
    p, y = probs[(key, "test")]
    err = test.copy()
    err["prob"] = p
    err["pred"] = (p >= THRESHOLD).astype(int)
    err["gold"] = y
    fp = err[(err.gold == 0) & (err.pred == 1)].sort_values("prob", ascending=False).head(5)
    fn = err[(err.gold == 1) & (err.pred == 0)].sort_values("prob").head(5)
    print(f"\n=== {key} === FP={((err.gold==0)&(err.pred==1)).sum()} FN={((err.gold==1)&(err.pred==0)).sum()}")
    display(fp[["comment_text", "prob"]].assign(comment_text=lambda d: d.comment_text.str.slice(0, 160)))
    display(fn[["comment_text", "prob"]].assign(comment_text=lambda d: d.comment_text.str.slice(0, 160)))


## 5. Fairness (identity FPR) — expect high FPR if degenerate


In [ ]:
probe = build_identity_probe_df()
# Subsample probe in quick mode (decoder is slow)
if QUICK_MODE and len(probe) > 60:
    probe = probe.sample(60, random_state=SEED).reset_index(drop=True)
fair_rows = []
for key, scorer in scorers.items():
    print(f"Fairness scoring {key} ...")
    pp = scorer.predict_toxic_proba(probe["text"].tolist())
    g = per_group_fpr(probe, pp, threshold=THRESHOLD)
    ctrl = control_fpr(scorer.predict_toxic_proba(CONTROL_TEXTS), threshold=THRESHOLD)
    fair_rows.append({
        "model": key,
        "overall_fpr": float(g["fpr"].mean()),
        "control_fpr": ctrl["fpr"],
        "max_group_fpr": float(g["fpr"].max()),
    })
fair_df = pd.DataFrame(fair_rows)
display(fair_df.round(3))
sns.barplot(data=fair_df, x="model", y="overall_fpr", color="crimson")
plt.title("Decoder identity FPR"); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()


## 6. Robustness (small toxic sample)


In [ ]:
toxic_pool = get_split_frame("test")
toxic_pool = toxic_pool[toxic_pool["toxic"] == 1]
if len(toxic_pool) > ROBUST_N:
    toxic_pool = toxic_pool.sample(ROBUST_N, random_state=SEED)
base_texts = toxic_pool["comment_text"].astype(str).tolist()
# Focus on a few strong obfuscations for speed
pert_names = ["leetspeak", "typos", "space_out"]
rob_rows = []
for key, scorer in scorers.items():
    print(f"Robustness {key} ...")
    base_p = np.asarray(scorer.predict_toxic_proba(base_texts))
    base_flag = base_p >= THRESHOLD
    for pname in pert_names:
        fn = PERTURBATIONS[pname]
        pert = [fn(t) for t in base_texts]
        pp = np.asarray(scorer.predict_toxic_proba(pert))
        still = pp >= THRESHOLD
        idx = np.where(base_flag)[0]
        evasion = float((~still[idx]).mean()) if len(idx) else float("nan")
        rob_rows.append({"model": key, "perturbation": pname, "evasion_rate": evasion,
                         "baseline_detection": float(base_flag.mean())})
rob_df = pd.DataFrame(rob_rows)
display(rob_df.round(3))
sns.barplot(data=rob_df, x="model", y="evasion_rate", hue="perturbation")
plt.title("Decoder evasion"); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()


## 7. Zero-shot vs FT gap (fills when d1-ft exists)


In [ ]:
if {"distilgpt2-d1", "distilgpt2-d1-ft"} <= set(scorers):
    a = metrics_df[(metrics_df.model == "distilgpt2-d1") & (metrics_df.split == "test")].iloc[0]
    b = metrics_df[(metrics_df.model == "distilgpt2-d1-ft") & (metrics_df.split == "test")].iloc[0]
    gap = pd.DataFrame([
        {"metric": m, "zero_shot": a[m], "fine_tuned": b[m], "delta": b[m] - a[m]}
        for m in ("f1", "roc_auc", "pr_auc", "accuracy", "ms_per_comment")
    ])
    display(gap.round(3))
    print("Positive delta on F1/ROC-AUC => FT helped.")
else:
    print("Waiting on distilgpt2-d1-ft weights — re-run this notebook after Colab 10.")


## 8. Save JSON


In [ ]:
out = {
    "seed": SEED,
    "quick_mode": QUICK_MODE,
    "eval_n": EVAL_N,
    "keys": KEYS,
    "loaded": list(scorers),
    "metrics": metric_rows,
    "fairness": fair_rows,
    "robustness": rob_df.to_dict(orient="records"),
    "d1_ft_ready": "distilgpt2-d1-ft" in scorers,
}
path = PROCESSED_DIR / "decoder_rigorous_audit.json"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2), encoding="utf-8")
print("Wrote", path)
